## Configuration and Imports

In [ ]:
import re
from datasets import load_dataset

# --- CONFIGURATION ---
DATASET_PATH = "microsoft/orca-math-word-problems-200k"
DATASET_SPLIT = "train"
SAMPLE_SIZE = 100
RANDOM_SEED = 42


## Extract Model Return

In [ ]:
def harmonize_orca_answer(orca_text: str) -> str:
    """Extracts the final number and appends the GSM8K delimiter format."""
    # Regex to find numbers, including negatives, commas, and decimals
    numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", str(orca_text))
    if numbers:
        final_number = numbers[-1].replace(",", "")
        if final_number.endswith("."):
            final_number = final_number[:-1]
        return f"{orca_text}\n#### {final_number}"
    return orca_text


def verify_harmonization():
    print(f"Loading {SAMPLE_SIZE} random samples from Orca Math...")
    # Load dataset
    dataset = load_dataset(DATASET_PATH, split=DATASET_SPLIT)

    # Shuffle with a seed to get a consistent random assortment of problem types
    sample_data = dataset.shuffle(seed=RANDOM_SEED).select(range(SAMPLE_SIZE))

    print(f"\n{'=' * 80}")
    print(f" DISPLAYING {SAMPLE_SIZE} HARMONIZED EXAMPLES")
    print(f"{'=' * 80}\n")

    failure_count = 0

    for i, example in enumerate(sample_data):
        question = example["question"]
        original_answer = example["answer"]
        harmonized_answer = harmonize_orca_answer(original_answer)

        # Check if our regex successfully appended the delimiter
        if "####" not in harmonized_answer:
            failure_count += 1

        print(f"--- EXAMPLE {i + 1} ---")
        print(f"QUESTION:\n{question}\n")
        print(f"HARMONIZED ANSWER:\n{harmonized_answer}")
        print(f"\n{'=' * 80}\n")

    print(f"Verification Complete!")
    print(f"Total Extractions Failed: {failure_count} out of {SAMPLE_SIZE}")

    if failure_count > 0:
        print("Note: Extractions fail if the text does not contain any numbers.")


## Execution

In [ ]:
if __name__ == "__main__":
    verify_harmonization()
